# Toto 2.0 — DIMER forecasting tutorial

**Profile:** `TASK-INFERENCE`  
**Capability:** zero-shot multivariate probabilistic forecasting with the pinned Toto 2.0 2.5B checkpoint

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API rather than reimplementing model inference. The default sample is demonstration evidence, not a production-quality or benchmark claim.

**Learning objectives:** resolve the immutable upstream model revision, validate a public/default input, run the supported task, inspect task-appropriate outputs, exercise an optional BYOD path, and export machine-readable outputs plus provenance.


## Prerequisites

Run in a fresh supported runtime. Install dependencies before importing PyTorch or Transformers. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.


## 1. Bootstrap the repository and pinned runtime

The installation cell installs this repository and its exact model-facing dependency versions. If installation replaces a pre-imported core framework, restart the runtime before continuing.

In [ ]:
%pip install -q -e .
import platform, numpy as np, pandas as pd, torch
print({'python':platform.python_version(),'torch':torch.__version__,'cuda':torch.cuda.is_available()})
assert torch.cuda.is_available(), 'A CUDA GPU is required for the Toto 2.5B release-reference path.'

## 2. Create deterministic sample or optional BYOD

The default two-variate sample is generated reproducibly. BYOD expects `timestamp` plus numeric target columns and is gated off by default. Missing values are rejected by this initial serving contract rather than silently imputed.

In [ ]:
USE_BYOD=False
if USE_BYOD:
    from google.colab import files
    name=next(iter(files.upload())); df=pd.read_csv(name); assert 'timestamp' in df.columns; values=df.drop(columns=['timestamp']).to_numpy(float).T
else:
    rng=np.random.default_rng(11); t=np.arange(320); v1=0.01*t+np.sin(t/9)+rng.normal(0,0.04,len(t)); v2=0.5*v1+np.cos(t/13)+rng.normal(0,0.04,len(t)); values=np.vstack([v1,v2])
assert np.isfinite(values).all()
print({'shape':values.shape,'sample':'BYOD' if USE_BYOD else 'deterministic synthetic'})

## 3. Chronological holdout and naive baseline

The final horizon is withheld from model context. This enforces chronological evaluation and prevents future-target leakage.

In [ ]:
from toto_forecasting_pipeline import TotoForecastPipeline, last_value_baseline, mae, rmse, interval_coverage, MODEL_ID, MODEL_REVISION
horizon=48; context=values[:,:-horizon]; truth=values[:,-horizon:]; baseline=last_value_baseline(context,horizon)
print({'model_id':MODEL_ID,'revision':MODEL_REVISION,'context':context.shape[1],'horizon':horizon,'baseline_mae':mae(truth,baseline)})

## 4. Resolve Toto 2.0 and forecast

Toto returns q=0.1–0.9. The q=0.5 value is the median point forecast; q=0.1/q=0.9 are model quantiles, not guaranteed confidence bounds.

In [ ]:
pipe=TotoForecastPipeline.from_pretrained(device='cuda')
result=pipe.forecast(context,horizon=horizon,decode_block_size=768)
pred=result['median']; lo=result['quantiles'][:,0,:]; hi=result['quantiles'][:,-1,:]
metrics={'mae':mae(truth,pred),'rmse':rmse(truth,pred),'baseline_mae':mae(truth,baseline),'baseline_rmse':rmse(truth,baseline),'q10_q90_empirical_coverage':interval_coverage(truth,lo,hi)}
print(metrics)

## 5. Export forecasts and provenance

Machine-readable outputs preserve variate/time-step alignment, truth for this holdout, baseline, all quantiles, model revision, and decode strategy.

In [ ]:
import json, os
os.makedirs('outputs',exist_ok=True)
rows=[]
for v in range(pred.shape[0]):
    for h in range(horizon):
        r={'variate':v,'step':h+1,'truth':float(truth[v,h]),'baseline':float(baseline[v,h])}
        for qi,qlevel in enumerate(result['quantile_levels']): r[f'q{int(qlevel*100):02d}']=float(result['quantiles'][v,qi,h])
        rows.append(r)
pd.DataFrame(rows).to_csv('outputs/toto_forecast.csv',index=False)
prov={'model_id':MODEL_ID,'model_revision':MODEL_REVISION,'metrics':metrics,'context_length':context.shape[1],'horizon':horizon,'decode_block_size':result['decode_block_size'],'runtime':{'python':platform.python_version(),'torch':torch.__version__,'device':pipe.device}}
with open('outputs/toto_provenance.json','w') as f: json.dump(prov,f,indent=2)
print(['outputs/toto_forecast.csv','outputs/toto_provenance.json'])

## Interpretation and limits

No gradient training or fine-tuning occurs. The tutorial's error and empirical q10–q90 coverage are tied to one synthetic chronological holdout. Model quantiles must not be described as guaranteed confidence intervals. Toto 2.0 exogenous-variable support and fine-tuning are deliberately excluded because they are not part of the current upstream 2.0 inference release.

Successful execution proves that this repository revision can acquire the pinned model, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

## References

- Upstream model: https://huggingface.co/Datadog/Toto-2.0-2.5B
- Upstream code: https://github.com/DataDog/toto
- Technical report: https://arxiv.org/abs/2605.20119
